# CAN-VAE — Metrics (3 heads)

Evaluate a model trained with:

```bash
cd "Python scripts"
python -m datasets.can.preprocess
python -m datasets.can.preprocess --include-attacks
python -m experiments.train --dataset can --model can_vae --epochs 50 --objective recon
```

This notebook:
1. Loads the trained `can_vae` bundle + training scaler
2. Builds message windows from engineered HCRL parquet (same FE as `can_gru`)
3. Scores **real** attack traces (DoS / Fuzzy / RPM / gear) with all **three** heads:
   reconstruction / latent distance / latent clustering
4. Reports PR AUC, macro F1, AUROC, recall/FPR, per-attack breakdown, latency

> Fair comparison with `evaluate_can_gru.ipynb`: same profile, windows, and attacks.

## 1. Configuration

In [ ]:
from pathlib import Path

BUNDLE_DIR = Path("experiments/checkpoints/can")
NAME = "can_vae_can"  # trainer default: {model}_{dataset}

PROCESSED_DIR = Path("../Data/CAN/processed")

MAX_BENIGN_WINDOWS = 10_000
MAX_ATTACK_WINDOWS_PER_KIND = 5_000
ATTACK_KINDS = ("dos", "fuzzy", "rpm", "gear")

SEED = 42
FIGURES_DIR = Path("experiments/figures/can")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Imports & load the trained bundle

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)

_HERE = Path.cwd().resolve()
for candidate in (_HERE, _HERE.parent, _HERE.parent.parent):
    if (candidate / "ids_core").is_dir() and (candidate / "models").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        SCRIPTS = candidate
        break
else:
    raise SystemExit("Could not find Python scripts/ (ids_core + models) on path")

from ids_core.data import make_windows
from ids_core.profiles import get_profile
from models.can_gru.features import FEATURES, N_RAW_FEATURES
from models.can_vae import CANLSTMVAE

np.random.seed(SEED)
rng = np.random.default_rng(SEED)

profile = get_profile("can")
BUNDLE_DIR = (SCRIPTS / BUNDLE_DIR).resolve() if not BUNDLE_DIR.is_absolute() else BUNDLE_DIR
PROCESSED_DIR = (
    (SCRIPTS / PROCESSED_DIR).resolve()
    if not Path(PROCESSED_DIR).is_absolute()
    else Path(PROCESSED_DIR)
)
FIGURES_DIR = (
    (SCRIPTS / FIGURES_DIR).resolve()
    if not FIGURES_DIR.is_absolute()
    else FIGURES_DIR
)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

meta_path = BUNDLE_DIR / f"{NAME}_meta.json"
scaler_path = BUNDLE_DIR / f"{NAME}_scaler.npz"
ckpt = BUNDLE_DIR / f"{NAME}_params.msgpack"
model_meta = BUNDLE_DIR / f"{NAME}_model_meta.json"
for p in (ckpt, scaler_path, model_meta):
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing {p}. Train first:\n"
            f"  python -m experiments.train --dataset can --model can_vae --epochs 50"
        )

trainer_meta = json.loads(meta_path.read_text()) if meta_path.is_file() else {}
WINDOW = int(trainer_meta.get("window_size", profile.window_size))
STRIDE = int(trainer_meta.get("stride", profile.stride))

scaler = np.load(scaler_path)
MU = scaler["mean"].astype(np.float32)
SD = scaler["std"].astype(np.float32)
SD = np.where(SD > 1e-6, SD, 1.0).astype(np.float32)

model = CANLSTMVAE.load(str(BUNDLE_DIR), name=NAME)
WINDOW = int(model.cfg.window)

n_params = model.num_params()
print(f"Loaded bundle: {BUNDLE_DIR} ({NAME})")
print(f"  features     : {N_RAW_FEATURES} → model {model.cfg.n_model_features} after ID embed")
print(f"  window/stride: {WINDOW}/{STRIDE}")
print(f"  parameters   : {n_params:,}")
print(f"  recon thr    : {model.recon_threshold:.6f}")
print(f"  dist thr     : {model.dist_threshold:.6f}")
print(f"  processed    : {PROCESSED_DIR}")

## 3. Load engineered parquet & build windows

Uses the **training** scaler. Attack windows require:

```bash
python -m datasets.can.preprocess --include-attacks
```

In [ ]:
def load_engineered(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise SystemExit(f"{path.name} missing columns: {missing}")
    if "is_anomaly" not in df.columns:
        df["is_anomaly"] = 0
    return df


def windows_from_df(
    df: pd.DataFrame,
    *,
    max_windows: int | None = None,
    force_label: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    feats = df.loc[:, list(FEATURES)].to_numpy(dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    norm = ((feats - MU) / SD).astype(np.float32)
    row_labels = df["is_anomaly"].to_numpy(dtype=np.int8)
    w, y, _ = make_windows(norm, stride=STRIDE, window=WINDOW, labels=row_labels)
    if force_label is not None and len(y):
        y = np.full(len(y), force_label, dtype=np.int8)
    if max_windows is not None and len(w) > max_windows:
        idx = np.linspace(0, len(w) - 1, max_windows, dtype=int)
        w, y = w[idx], y[idx]
    return w, y


normal_path = PROCESSED_DIR / "normal.parquet"
if not normal_path.is_file():
    raise FileNotFoundError(
        f"Missing {normal_path}. Run: python -m datasets.can.preprocess"
    )

normal_df = load_engineered(normal_path)
benign_w, _ = windows_from_df(
    normal_df, max_windows=MAX_BENIGN_WINDOWS, force_label=0
)
print(f"[Benign] {normal_path.name}: rows={len(normal_df):,}  windows={len(benign_w):,}")

attack_parts, kind_parts = [], []
for kind in ATTACK_KINDS:
    path = PROCESSED_DIR / f"{kind}.parquet"
    if not path.is_file():
        print(f"[Skip] missing {path.name}")
        continue
    df = load_engineered(path)
    w, y = windows_from_df(df, max_windows=MAX_ATTACK_WINDOWS_PER_KIND)
    if (df["is_anomaly"] == 1).any():
        keep = y == 1
        w, y = w[keep], y[keep]
    else:
        y = np.ones(len(w), dtype=np.int8)
    print(f"[Attack] {path.name}: rows={len(df):,}  attack_windows={len(w):,}")
    if len(w):
        attack_parts.append(w)
        kind_parts.append(np.full(len(w), kind, dtype=object))

if not attack_parts:
    raise SystemExit("No attack parquet. Run: python -m datasets.can.preprocess --include-attacks")

attack_w = np.concatenate(attack_parts, axis=0)
attack_kinds = np.concatenate(kind_parts, axis=0)

windows = np.concatenate([benign_w, attack_w], axis=0)
labels = np.concatenate(
    [np.zeros(len(benign_w), dtype=np.int8), np.ones(len(attack_w), dtype=np.int8)],
    axis=0,
)
kinds = np.concatenate(
    [np.full(len(benign_w), "benign", dtype=object), attack_kinds], axis=0
)

perm = rng.permutation(len(windows))
windows, labels, kinds = windows[perm], labels[perm], kinds[perm]
prevalence = float(labels.mean())
print(
    f"[Test] benign={int((labels == 0).sum()):,}  attack={int((labels == 1).sum()):,}"
    f"  prevalence={prevalence:.3f}"
)
print(
    "[Attacks]",
    ", ".join(f"{k}={int((kinds == k).sum())}" for k in ATTACK_KINDS if (kinds == k).any()),
)

## 4. Score all three heads

- **Reconstruction** — MSE in embedded feature space; threshold = calibrated recon τ
- **Latent distance** — NN Euclidean distance in z; threshold = calibrated dist τ
- **Latent clustering** — Isolation Forest score / hard labels

In [ ]:
recon_scores = model.reconstruction_error(windows)
recon_pred = (recon_scores > model.recon_threshold).astype(int)

dist_scores = model.latent_distance(windows)
dist_pred = (dist_scores > model.dist_threshold).astype(int)

clus_scores = model.latent_cluster_score(windows)
clus_pred = model.latent_cluster_predict(windows)

HEADS = {
    "Reconstruction": (recon_scores, recon_pred, float(model.recon_threshold)),
    "Latent distance": (dist_scores, dist_pred, float(model.dist_threshold)),
    "Latent clustering": (clus_scores, clus_pred, None),
}
print(f"scored {len(windows):,} windows")

## 5. Detection latency

Streaming protocol: one window per call → p50 / p99 (after JIT warm-up).

In [ ]:
score_fns = {
    "Reconstruction": model.reconstruction_error,
    "Latent distance": model.latent_distance,
    "Latent clustering": model.latent_cluster_score,
}

n_lat = min(len(windows), 1000)
latency = {}
for name, fn in score_fns.items():
    _ = fn(windows[:64])  # warm-up
    lat = np.empty(n_lat, dtype=np.float32)
    for i in range(n_lat):
        t0 = time.perf_counter()
        fn(windows[i : i + 1])
        lat[i] = (time.perf_counter() - t0) * 1e6
    latency[name] = lat
    print(
        f"{name:20s}  p50={np.percentile(lat, 50):.1f}µs  "
        f"p99={np.percentile(lat, 99):.1f}µs"
    )

heads = list(latency)
p50s = [float(np.percentile(latency[h], 50)) for h in heads]
p99s = [float(np.percentile(latency[h], 99)) for h in heads]
x = np.arange(len(heads))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width / 2, p50s, width, label="p50", color="#41b6c4")
ax.bar(x + width / 2, p99s, width, label="p99", color="#225ea8")
ax.set_xticks(x, heads, rotation=15, ha="right")
ax.set_ylabel("µs / window")
ax.set_title("CAN-VAE streaming latency by head")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "can_vae_latency.png", dpi=120)
plt.show()

print(f"parameters: {n_params:,}  (~{n_params * 4 / 1024:.1f} KiB float32)")

## 6. Headline metrics

In [ ]:
rows = []
for name, (scores, pred, thr) in HEADS.items():
    rows.append({
        "head": name,
        "PR AUC": average_precision_score(labels, scores),
        "macro F1": f1_score(labels, pred, average="macro", zero_division=0),
        "AUROC": roc_auc_score(labels, scores),
        "recall": float(pred[labels == 1].mean()) if (labels == 1).any() else float("nan"),
        "FPR": float(pred[labels == 0].mean()) if (labels == 0).any() else float("nan"),
        "threshold": thr if thr is not None else float("nan"),
        "latency p50 (µs)": float(np.percentile(latency[name], 50)),
        "latency p99 (µs)": float(np.percentile(latency[name], 99)),
    })
summary = pd.DataFrame(rows).set_index("head")
print(f"random-baseline PR AUC = prevalence = {prevalence:.3f}\n")
display(summary.round(3))
summary.to_csv(FIGURES_DIR / "can_vae_summary.csv")

## 7. Precision–Recall curves (all heads)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, (scores, pred, thr) in HEADS.items():
    prec, rec, _ = precision_recall_curve(labels, scores)
    ap = average_precision_score(labels, scores)
    ax.plot(rec, prec, lw=2, label=f"{name} (PR AUC={ap:.3f})")
ax.axhline(prevalence, ls="--", color="gray", lw=1, label=f"baseline={prevalence:.3f}")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("Precision–Recall — CAN-VAE (3 heads)")
ax.legend(loc="lower left")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "can_vae_pr_curve.png", dpi=120)
plt.show()

## 8. Score distributions (benign vs attack)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, (scores, pred, thr)) in zip(axes, HEADS.items()):
    b = scores[labels == 0]
    a = scores[labels == 1]
    lo, hi = np.percentile(scores, 1), np.percentile(scores, 99)
    bins = np.linspace(lo, hi, 60)
    ax.hist(b, bins=bins, alpha=0.6, density=True, color="#2c7fb8", label="benign")
    ax.hist(a, bins=bins, alpha=0.6, density=True, color="#d95f0e", label="attack")
    if thr is not None:
        ax.axvline(thr, color="red", ls="--", lw=2, label="threshold")
    ax.set_title(name)
    ax.set_xlabel("score")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "can_vae_score_hist.png", dpi=120)
plt.show()

## 9. Per-attack PR AUC (each family vs benign)

In [ ]:
rows = []
for k in ATTACK_KINDS:
    m = (kinds == k) & (labels == 1)
    if not m.any():
        continue
    row = {"attack": k, "n_windows": int(m.sum())}
    for name, (scores, pred, thr) in HEADS.items():
        benign_s = scores[labels == 0]
        y = np.concatenate([np.zeros(len(benign_s)), np.ones(int(m.sum()))])
        s = np.concatenate([benign_s, scores[m]])
        row[name] = average_precision_score(y, s)
    rows.append(row)

per_attack = pd.DataFrame(rows).set_index("attack")
display(per_attack.round(3))

metric_cols = [c for c in per_attack.columns if c != "n_windows"]
x = np.arange(len(per_attack))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 4.5))
for i, col in enumerate(metric_cols):
    ax.bar(x + (i - 1) * width, per_attack[col], width, label=col)
ax.axhline(prevalence, color="gray", ls="--", lw=1, label=f"baseline={prevalence:.3f}")
ax.set_xticks(x, per_attack.index)
ax.set_ylim(0, 1.05)
ax.set_ylabel("PR AUC vs benign")
ax.set_title("Per-attack detectability — CAN-VAE")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "can_vae_per_attack.png", dpi=120)
plt.show()

per_attack.to_csv(FIGURES_DIR / "can_vae_per_attack.csv")

## 10. Reading the results

- Compare **Reconstruction** PR AUC / F1 / latency directly to `evaluate_can_gru.ipynb`.
- **Latent distance / clustering** use encoder only at inference — usually cheaper than full recon.
- Train with `--objective latent` (higher β) if you want to emphasise the latent heads.

Figures → `experiments/figures/can/can_vae_*`.